In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from datetime import datetime

In [2]:
df = pd.read_csv('requests_raw.csv', encoding='latin-1',index_col = False)

In [3]:
df

,request_id,listing_id,requester_id,owner_id,request_type,status,requested_at,responded_at,handover_date,due_date,returned_at,handover_location,message
0,R000623,L01041,U0186,U0085,receive,completed,"06 Jun 2025, 10:06 AM",2025-06-06 12:53:19,2025-06-06,NaN,NaN,NaN,"Mujhe ye chahiye, exam hai next week. Please d..."
1,R001312,L02558,U0168,U0487,borrow,cancelled,2024-10-31 20:39:32,2024-11-03 12:23:42,NaN,NaN,NaN,NaN,"Please give me this, I will return on time and..."
2,R005128,L00489,U0563,U0682,receive,completed,2026-05-05 08:03:03,05/05/2026 08:50,07/05/2026,NaN,NaN,NaN,Need for lab practical next week.
3,R000420,L02828,U0194,U0007,exchange,completed,2025-07-30 02:05:54,"30 Jul 2025, 02:38 PM",2025-08-02,NaN,NaN,Canteen,Is this still available? I need it urgently.
4,R001433,L02347,U0074,U0256,BORROW,accepted,2026-08-10 06:13:43,2026-08-11 06:13:34,2026-08-14,12 Nov 2026,NaN,Library,"Hi, could I pick this up tomorrow after classes?"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6931,R002182,L02917,U0478,U0573,receive,completed,19/05/2025 22:18,2025-05-20 12:03:39,2025-05-23,NaN,NaN,Library,Bhai ye available hai kya? mujhe kal tak chahiye.
6932,R002410,L01498,U0405,U0253,BORROW,accepted,"20 Sep 2026, 08:30 PM",2026-09-22 14:00:29,2026-09-24,2026-11-23,NaN,Hostel gate,NaN
6933,R002034,L00703,U0176,U0305,donation,rejected,06/08/2024 08:59,06/08/2024 17:27,NaN,NaN,NaN,NaN,Need for lab practical next week.
6934,R001365,L00385,U0305,U0613,borrow request,rejected,"30 Jul 2026, 04:39 PM",31/07/2026 01:39,NaN,NaN,NaN,NaN,"Hello, interested in this item. Please let me ..."


In [4]:
df.shape

(6936, 13)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6936 entries, 0 to 6935
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   request_id         6936 non-null   str  
 1   listing_id         6936 non-null   str  
 2   requester_id       6936 non-null   str  
 3   owner_id           6936 non-null   str  
 4   request_type       6775 non-null   str  
 5   status             6936 non-null   str  
 6   requested_at       6936 non-null   str  
 7   responded_at       6724 non-null   str  
 8   handover_date      4255 non-null   str  
 9   due_date           1126 non-null   str  
 10  returned_at        1018 non-null   str  
 11  handover_location  3393 non-null   str  
 12  message            5918 non-null   str  
dtypes: str(13)
memory usage: 1.5 MB


In [6]:
df.isnull().sum()

request_id              0
listing_id              0
requester_id            0
owner_id                0
request_type          161
status                  0
requested_at            0
responded_at          212
handover_date        2681
due_date             5810
returned_at          5918
handover_location    3543
message              1018
dtype: int64

In [7]:
df.nunique()

request_id           6800
listing_id           2563
requester_id          833
owner_id              797
request_type           14
status                 27
requested_at         6800
responded_at         6590
handover_date        1910
due_date              834
returned_at           761
handover_location       5
message                14
dtype: int64

In [8]:
df.duplicated().sum()

np.int64(136)

In [9]:
df = df.drop_duplicates().reset_index(drop=True)

In [10]:
df['request_id'].duplicated().sum()

np.int64(0)

In [11]:
df.columns

Index(['request_id', 'listing_id', 'requester_id', 'owner_id', 'request_type',
       'status', 'requested_at', 'responded_at', 'handover_date', 'due_date',
       'returned_at', 'handover_location', 'message'],
      dtype='str')

In [12]:
df.isnull().sum()

request_id              0
listing_id              0
requester_id            0
owner_id                0
request_type          157
status                  0
requested_at            0
responded_at          208
handover_date        2627
due_date             5702
returned_at          5802
handover_location    3477
message               999
dtype: int64

In [13]:
df['request_type'] = df['request_type'].fillna(df['request_type'].mode()[0])

In [14]:
df['request_type'] = df['request_type'].str.strip()

In [15]:
df['request_type'] = df['request_type'].str.lower()



In [16]:
df['request_type'].unique()

<ArrowStringArray>
['receive', 'borrow', 'exchange', 'lend', 'donation', 'borrow request',
 'swap']
Length: 7, dtype: str

In [17]:
df['request_type'] = df['request_type'].replace({
    'lend' : 'borrow',
    'swap' : 'exchange',
    
})

In [18]:
df.isnull().sum()

request_id              0
listing_id              0
requester_id            0
owner_id                0
request_type            0
status                  0
requested_at            0
responded_at          208
handover_date        2627
due_date             5702
returned_at          5802
handover_location    3477
message               999
dtype: int64

In [19]:
df['responded_at']

0         2025-06-06 12:53:19
1         2024-11-03 12:23:42
2            05/05/2026 08:50
3       30 Jul 2025, 02:38 PM
4         2026-08-11 06:13:34
                ...          
6795      2025-05-20 12:03:39
6796      2026-09-22 14:00:29
6797         06/08/2024 17:27
6798         31/07/2026 01:39
6799      2026-02-28 23:32:56
Name: responded_at, Length: 6800, dtype: str

In [20]:
df['responded_at'] = pd.to_datetime(
    df['responded_at'],
    format='mixed',
    dayfirst=True,
    errors='coerce'
)

In [21]:
df['responded_at'].isna().sum()

np.int64(208)

In [22]:
df['responded_at']

0      2025-06-06 12:53:19
1      2024-03-11 12:23:42
2      2026-05-05 08:50:00
3      2025-07-30 14:38:00
4      2026-11-08 06:13:34
               ...        
6795   2025-05-20 12:03:39
6796   2026-09-22 14:00:29
6797   2024-08-06 17:27:00
6798   2026-07-31 01:39:00
6799   2026-02-28 23:32:56
Name: responded_at, Length: 6800, dtype: datetime64[us]

In [23]:
df['responded_at'] = df['responded_at'].fillna(df['responded_at'].mean())

In [24]:
df['responded_at'].isnull().sum()

np.int64(0)

In [25]:
df.isnull().sum()

request_id              0
listing_id              0
requester_id            0
owner_id                0
request_type            0
status                  0
requested_at            0
responded_at            0
handover_date        2627
due_date             5702
returned_at          5802
handover_location    3477
message               999
dtype: int64

In [26]:
df['handover_date'].unique()

<ArrowStringArray>
[  '2025-06-06',            nan,   '07/05/2026',   '2025-08-02',
   '2026-08-14',   '2025-08-12',   '2024-09-14',   '15/07/2024',
   '2025-05-06',   '2025-08-28',
 ...
   '2024-11-08',     '09-12-24', 'Mar 05, 2026',   '03/04/2026',
 'Dec 03, 2024', 'Aug 19, 2026',   '29/06/2025',  '12 Mar 2026',
   '17/03/2025', 'Apr 16, 2026']
Length: 1911, dtype: str

In [27]:
df['handover_date'] = pd.to_datetime(
    df['handover_date'],
    format='mixed',
    dayfirst=True,
    errors='coerce'
)

In [28]:
df['handover_date'] = df['handover_date'].fillna(df['handover_date'].mean())

In [29]:
df['due_date']

0               NaN
1               NaN
2               NaN
3               NaN
4       12 Nov 2026
           ...     
6795            NaN
6796     2026-11-23
6797            NaN
6798            NaN
6799            NaN
Name: due_date, Length: 6800, dtype: str

In [31]:
df.to_csv("cleaned_request_data.csv",index=False)